# 25. LightGBM & CatBoost — `lightgbm · catboost`
**Python Libraries Lab · Phase 2 · Machine Learning**

_Faster boosting, and boosting that understands categories._

✅ **Runs in Colab** as-is.

LightGBM and CatBoost are gradient boosting libraries, just like XGBoost, built by Microsoft and Yandex. LightGBM grows trees in a clever leaf by leaf way that makes it very fast on big data. CatBoost handles text categories, like city or product type, directly and carefully, so you often get great results with almost no tuning.

**In this notebook:** You will predict whether a ShopEasy order will be returned, using data with several text categories. You will train LightGBM and CatBoost, compare them with XGBoost on the same split, and measure speed and accuracy using AUC.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q lightgbm catboost
import importlib.metadata as md
for pkg in ['lightgbm', 'catboost']:
    print(pkg, md.version(pkg))

## Step 1 · Data with text categories
Predict `returned` (1 = customer sent the item back).

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(1)
n = 8000
df = pd.DataFrame({
    "category": rng.choice(["Books", "Electronics", "Clothing", "Home", "Toys", "Beauty"], n),
    "brand_tier": rng.choice(["budget", "mid", "premium"], n),
    "city": rng.choice(["Mumbai", "Delhi", "Pune", "Bengaluru", "Chennai"], n),
    "price": rng.choice([199, 499, 999, 1499, 2999, 4999], n),
    "delivery_days": rng.integers(1, 9, n),
    "first_order": rng.integers(0, 2, n),
})
logit = (-2.5 + 1.2 * (df["category"] == "Clothing") + 0.5 * (df["category"] == "Electronics")
         + 0.25 * df["delivery_days"] - 0.4 * (df["brand_tier"] == "premium") + 0.5 * df["first_order"])
df["returned"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
cat_cols = ["category", "brand_tier", "city"]
for c in cat_cols:
    df[c] = df[c].astype("category")
print("Return rate:", df["returned"].mean().round(3))
df.head()

## Step 2 · One split for a fair comparison
All three models use exactly the same rows.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import time

X, y = df.drop(columns="returned"), df["returned"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
results = {}

## Step 3 · LightGBM
Pandas `category` columns are used directly.

In [ ]:
import lightgbm as lgb

t = time.perf_counter()
lgbm = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1, random_state=0)
lgbm.fit(X_train, y_train)
results["LightGBM"] = (roc_auc_score(y_test, lgbm.predict_proba(X_test)[:, 1]), time.perf_counter() - t)
print("LightGBM AUC %.3f in %.2fs" % results["LightGBM"])

## Step 4 · CatBoost
Tell CatBoost which columns are categories; it handles the rest.

In [ ]:
from catboost import CatBoostClassifier

t = time.perf_counter()
cat = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6, verbose=0, random_seed=0, cat_features=cat_cols)
cat.fit(X_train.astype({c: str for c in cat_cols}), y_train)
results["CatBoost"] = (roc_auc_score(y_test, cat.predict_proba(X_test.astype({c: str for c in cat_cols}))[:, 1]), time.perf_counter() - t)
print("CatBoost AUC %.3f in %.2fs" % results["CatBoost"])

## Step 5 · XGBoost for comparison
XGBoost also accepts category columns with `enable_categorical=True`.

In [ ]:
import xgboost as xgb

t = time.perf_counter()
xg = xgb.XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, enable_categorical=True, tree_method="hist", random_state=0)
xg.fit(X_train, y_train)
results["XGBoost"] = (roc_auc_score(y_test, xg.predict_proba(X_test)[:, 1]), time.perf_counter() - t)

print(f"{'Model':<10} {'AUC':>6} {'Seconds':>8}")
for name, (auc, secs) in sorted(results.items(), key=lambda r: -r[1][0]):
    print(f"{name:<10} {auc:>6.3f} {secs:>8.2f}")

## Step 6 · What CatBoost learned
Feature importance shows category and delivery time drive returns, matching how we built the data.

In [ ]:
pd.Series(cat.get_feature_importance(), index=X.columns).sort_values(ascending=False).round(1)

## ✍️ Your turn
Train `lgb.LGBMClassifier(n_estimators=100, verbose=-1, random_state=0)` and store its **test AUC** in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    m = lgb.LGBMClassifier(n_estimators=100, verbose=-1, random_state=0).fit(X_train, y_train)
    exp = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    ok = answer is not None and abs(float(answer) - exp) < 1e-9
    return ok, f"Expected about {exp:.3f}, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
m = lgb.LGBMClassifier(n_estimators=100, verbose=-1, random_state=0).fit(X_train, y_train)
answer = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Leaf-wise growth**
- **Categorical features**
- **AUC**
- **Ordered boosting**
- **Model comparison**

**Next:** 26. TensorFlow & Keras (`tensorflow · keras`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/26_tensorflow_keras.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install lightgbm catboost
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.